In [1]:
# Upload files if running in Google Colab
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    pass


Saving 97.mat to 97.mat
Saving faulty.mat to faulty.mat


In [2]:
import numpy as np


DATA PROCESSING

In [3]:
import os
from scipy.io import loadmat
file_name = 'normal.mat' if os.path.exists('normal.mat') else '97.mat'
data = loadmat(file_name)
de_signal = data['X097_DE_time'].flatten()
print(de_signal)
window_size = 1024;
window = [];
de_signal_2d = [];
for i in range(0, len(de_signal), window_size):
    chunk = de_signal[i : i + window_size]
    if len(chunk) == window_size:
        de_signal_2d.append(chunk)
        chunk = []
    else:

        chunk = [] #discarded
de_signal_2d = np.array(de_signal_2d)
print(de_signal_2d.shape)


[ 0.05319692  0.08866154  0.09971815 ... -0.03463015  0.01668923
  0.04693846]
(238, 1024)


In [4]:
fft_windows = []
for window in de_signal_2d:
  fft_result = np.fft.fft(window)
  magnitude = np.abs(fft_result)
  fft_windows.append(magnitude)
fft_windows = np.array(fft_windows)
print(fft_windows.shape)

(238, 1024)


In [5]:
fft_windows_half = []
for window in fft_windows:
  fft_result = window[:len(window)//2]
  fft_windows_half.append(fft_result)
fft_windows_half = np.array(fft_windows_half)
print(fft_windows_half.shape)

(238, 512)


In [6]:
data_fault = loadmat('faulty.mat')
de_signal_fault = data_fault['X106_DE_time'].flatten()
window_size_fault = window_size
window_fault = []
de_signal_2d_fault = []
for i in range(0,len(de_signal_fault),1024):
  chunk_fault = de_signal_fault[i:i+1024]
  if(len(chunk_fault) == 1024):
    de_signal_2d_fault.append(chunk_fault)
    chunk_fault = []
  else:
    chunk_fault = []

In [7]:
de_signal_2d_fault = np.array(de_signal_2d_fault)
print(de_signal_2d_fault.shape)

(119, 1024)


In [8]:
fft_windows_fault = np.abs(np.fft.fft(de_signal_2d_fault, axis=1))
print(fft_windows_fault.shape)

(119, 1024)


In [9]:
fft_windows_half_fault = []
for window in fft_windows_fault:
    fft_result = window[:len(window)//2]
    fft_windows_half_fault.append(fft_result)
fft_windows_half_fault = np.array(fft_windows_half_fault)
print(fft_windows_half_fault.shape)

(119, 512)


In [10]:
fft_windows_half_re = fft_windows_half.reshape(-1,512,1)
print(fft_windows_half_re.shape)
fft_windows_half_fault_reshaped = fft_windows_half_fault.reshape(-1, fft_windows_half_fault.shape[1], 1)
print(fft_windows_half_fault_reshaped.shape)

(238, 512, 1)
(119, 512, 1)


TRAINING AND BUILDING OF THE MODEL

In [14]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers,models
input = layers.Input(shape = (512,1))
x = layers.Conv1D(filters=16, kernel_size=16, padding="same", activation="relu")(input)
x = layers.MaxPooling1D(pool_size=2, padding="same")(x)

x = layers.Conv1D(filters=32, kernel_size=8, padding="same", activation="relu")(x)
encoded = layers.MaxPooling1D(pool_size=2, padding="same")(x)

x = layers.Conv1D(filters=32, kernel_size=8, padding="same", activation="relu")(encoded)
x = layers.UpSampling1D(size=2)(x)

x = layers.Conv1D(filters=16, kernel_size=16, padding="same", activation="relu")(x)
x = layers.UpSampling1D(size=2)(x)

output = layers.Conv1D(filters=1,kernel_size=3,padding="same",activation = "linear")(x)
autoencoder = keras.Model(input,output)
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()
autoencoder.fit(fft_windows_half_re,fft_windows_half_re)
healthy_pred = autoencoder.predict(fft_windows_half_re)
healthy_errors = ((fft_windows_half_re - healthy_pred)**2).mean(axis=(1,2))
threshold =healthy_errors.max()
final = autoencoder.predict(fft_windows_half_fault_reshaped)
errors = ((final - fft_windows_half_fault_reshaped)**2)

window_errors = errors.mean(axis=(1,2))
predictions = window_errors > threshold
print(predictions)
most_anomalous_window = window_errors.argmax()

print("Most anomalous window:", most_anomalous_window)
print("Reconstruction error:", window_errors[most_anomalous_window])

# Save the trained autoencoder model
autoencoder.save('bearing_autoencoder.keras')
print("Model saved to bearing_autoencoder.keras")


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 512, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_15 (Conv1D)              │ (None, 512, 16)        │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_6 (MaxPooling1D)  │ (None, 256, 16)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_16 (Conv1D)              │ (None, 256, 32)        │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_7 (MaxPooling1D)  │ (None, 128, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_17 (Conv1D)              │ (None, 128, 32)        │         8,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling1d_6 (UpSampling1D)  │ (None, 256, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_18 (Conv1D)              │ (None, 256, 16)        │         8,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling1d_7 (UpSampling1D)  │ (None, 512, 16)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_19 (Conv1D)              │ (None, 512, 1)         │            49 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,881 (81.57 KB)

 Trainable params: 20,881 (81.57 KB)

 Non-trainable params: 0 (0.00 B)

8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - loss: 3.2211
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True]
Most anomalous window: 36
Reconstruction error: 44.17966664005601


EVALUATION

In [24]:
healthy_test = fft_windows_half_re
healthy_labels = np.zeros(len(healthy_test),dtype = int)
anomalous_test = fft_windows_half_fault_reshaped
anomalous_labels = np.ones(len(anomalous_test),dtype = int)
test_data = np.concatenate((healthy_test, anomalous_test), axis=0)
test_labels = np.concatenate((healthy_labels, anomalous_labels), axis=0)
indices = np.random.permutation(len(test_data))
test_data = test_data[indices]
test_labels = test_labels[indices]
predictions = autoencoder.predict(test_data)
reconstruction_errors = ((test_data - predictions)**2).mean(axis=(1,2))
predictions= (reconstruction_errors>threshold).astype(int)

# --- Evaluation Metrics ---
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

accuracy = accuracy_score(test_labels, predictions)
print(f"Threshold: {threshold:.4f}")
print(f"Evaluation Accuracy: {accuracy * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(test_labels, predictions, target_names=['Healthy', 'Faulty']))

# --- Visual Evaluation: Confusion Matrix & Reconstruction Error Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Confusion Matrix
cm = confusion_matrix(test_labels, predictions)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Healthy', 'Faulty']).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f"Confusion Matrix (Accuracy: {accuracy*100:.1f}%)")

# 2. Reconstruction Error Distribution vs Threshold
healthy_errs = reconstruction_errors[test_labels == 0]
faulty_errs = reconstruction_errors[test_labels == 1]
axes[1].hist(healthy_errs, bins=20, alpha=0.7, label=f'Healthy (n={len(healthy_errs)})', color='green')
axes[1].hist(faulty_errs, bins=25, alpha=0.7, label=f'Faulty (n={len(faulty_errs)})', color='red')
axes[1].axvline(threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold ({threshold:.2f})')
axes[1].set_xlabel('Reconstruction Error (MSE)')
axes[1].set_ylabel('Number of Windows')
axes[1].set_title('Reconstruction Error Distribution vs Threshold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
Threshold: 2.8771843207168994
Healthy errors:
1.5766982125947424 2.8771843207168994
Faulty errors:
28.750949688873433 44.17966664005601
